In [ ]:
import os
import csv
import re
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import (AutoModel, AutoTokenizer, AutoModelForSequenceClassification, BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, TrainerCallback, EarlyStoppingCallback)
from transformers.utils import logging
from datasets import Dataset, load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,matthews_corrcoef,precision_recall_fscore_support,classification_report,confusion_matrix,roc_auc_score,roc_curve,auc)
from scipy.special import softmax
import matplotlib.pyplot as plt
from tqdm import tqdm
warnings.filterwarnings("ignore", message="PyTorch is not compiled with NCCL support")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Set seed for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if using GPU
torch.backends.cudnn.deterministic = True  # Ensures deterministic behavior
torch.backends.cudnn.benchmark = False  # Disables the in-tensor fusion optimizations

ds = load_dataset("tasksource/wiki-hades")

In [ ]:
text_column = "replaced"
label_column = "hallucination"

model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
backbone = AutoModel.from_pretrained(model_name).eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
backbone.to(device)

def parse_spans(texts, labels):
    clean_texts, char_spans, out_labels = [], [], []

    for text, label in zip(texts, labels):
        match = re.search(r"===([^=]+)===", text)
        if not match:
            continue

        target = match.group(1)
        prefix = text[:match.start()]
        clean = prefix + target + text[match.end():]

        c_start = len(prefix)
        c_end = c_start + len(target)

        clean_texts.append(clean)
        char_spans.append((c_start, c_end))
        out_labels.append(int(label))

    return clean_texts, char_spans, out_labels

train_clean, train_spans, train_labels = parse_spans(
    ds["train"][text_column],
    ds["train"][label_column]
)

test_texts, test_spans, test_labels = parse_spans(
    ds["validation"][text_column],
    ds["validation"][label_column]
)

train_texts, val_texts, train_spans, val_spans, train_labels, val_labels = train_test_split(
    train_clean,
    train_spans,
    train_labels,
    test_size=0.15,
    stratify=train_labels,
    random_state=42
)

def compute_span_max_embeddings(
    texts,
    spans,
    labels,
    batch_size=32,
    max_length=512
):
    all_embeddings, all_labels = [], []

    for i in tqdm(range(0, len(texts), batch_size), desc="Span Max Embeddings"):
        batch_texts = texts[i:i+batch_size]
        batch_spans = spans[i:i+batch_size]
        batch_labels = labels[i:i+batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_offsets_mapping=True,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)
        offsets = encoded["offset_mapping"]

        with torch.no_grad():
            outputs = backbone(input_ids=input_ids, attention_mask=attention_mask)
            h = outputs.last_hidden_state  # [B, T, H]

        for b in range(len(batch_texts)):
            c_start, c_end = batch_spans[b]

            token_idxs = [
                idx for idx, (s, e) in enumerate(offsets[b])
                if s < c_end and e > c_start
            ]

            # Safety fallback
            if len(token_idxs) == 0:
                pooled = h[b, 0]  # CLS fallback
            else:
                pooled = torch.max(h[b, token_idxs], dim=0).values

            all_embeddings.append(pooled.cpu().numpy())
            all_labels.append(batch_labels[b])

    return np.vstack(all_embeddings), np.array(all_labels)

train_embeds, train_labels = compute_span_max_embeddings(
    train_texts, train_spans, train_labels
)

val_embeds, val_labels = compute_span_max_embeddings(
    val_texts, val_spans, val_labels
)

test_embeds, test_labels = compute_span_max_embeddings(
    test_texts, test_spans, test_labels
)

class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = torch.tensor(embeddings, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        return {
            'embedding': self.embeddings[idx],
            'label': self.labels[idx]
        }

train_dataset = EmbeddingDataset(train_embeds, train_labels)
val_dataset   = EmbeddingDataset(val_embeds, val_labels)
test_dataset  = EmbeddingDataset(test_embeds, test_labels)

# Collate function
def collate_fn(batch):
    embeddings = torch.stack([item['embedding'] for item in batch])  # [B, hidden_size]
    labels = torch.tensor([item['label'] for item in batch], dtype=torch.long)
    return {'embeddings': embeddings, 'labels': labels}

In [ ]:
import torch.nn.functional as F
import math
import gc
# Bayesian Linear Layer
class BayesianLinear(nn.Module):
    def __init__(self, in_features, out_features, prior_std=1.0):
        super().__init__()

        self.weight_mu = nn.Parameter(torch.zeros(out_features, in_features))
        self.weight_logvar = nn.Parameter(torch.full((out_features, in_features), -10.0))

        self.bias_mu = nn.Parameter(torch.zeros(out_features))
        self.bias_logvar = nn.Parameter(torch.full((out_features,), -10.0))

        self.prior_std = prior_std

    def forward(self, x):
        weight_std = torch.exp(0.5 * self.weight_logvar)
        bias_std = torch.exp(0.5 * self.bias_logvar)

        weight = self.weight_mu + weight_std * torch.randn_like(weight_std)
        bias = self.bias_mu + bias_std * torch.randn_like(bias_std)

        return F.linear(x, weight, bias)

    def kl_divergence(self):
        prior_var = self.prior_std ** 2

        weight_kl = 0.5 * torch.sum(
            (self.weight_mu ** 2 + torch.exp(self.weight_logvar)) / prior_var
            - 1.0
            - self.weight_logvar
            + math.log(prior_var)
        )

        bias_kl = 0.5 * torch.sum(
            (self.bias_mu ** 2 + torch.exp(self.bias_logvar)) / prior_var
            - 1.0
            - self.bias_logvar
            + math.log(prior_var)
        )

        return weight_kl + bias_kl

class BayesianMLP(nn.Module):
    def __init__(self, in_features, num_labels, expansion=4):
        super().__init__()

        hidden = in_features * expansion
        self.fc1 = BayesianLinear(in_features, hidden)
        self.fc2 = BayesianLinear(hidden, num_labels)

    def forward(self, x):
        x = F.gelu(self.fc1(x))
        return self.fc2(x)

    def kl_divergence(self):
        return self.fc1.kl_divergence() + self.fc2.kl_divergence()

class BayesianEmbeddingClassifier(nn.Module):
    def __init__(self, embedding_dim, num_labels=2, beta=1e-4):
        super().__init__()

        self.dropout = nn.Dropout(0.03)
        self.classifier = BayesianMLP(embedding_dim, num_labels, expansion=4)
        self.beta = beta

    def forward(self, embeddings=None, labels=None):
        x = self.dropout(embeddings)
        logits = self.classifier(x)

        loss = None
        if labels is not None:
            ce_loss = F.cross_entropy(logits, labels)
            kl_loss = self.classifier.kl_divergence() / embeddings.size(0)
            loss = ce_loss + self.beta * kl_loss

        return {"loss": loss, "logits": logits}

## modify from here dont touch above code ###
import numpy as np
import pandas as pd
import torch
import random
import gc
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    brier_score_loss,
    roc_auc_score,          # <-- Added
    average_precision_score # <-- Added (AUPRC)
)

# ---- ECE Calculation Function ----
def compute_ece(y_true, y_prob, n_bins=10):
    ece = 0.0
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    for i in range(n_bins):
        bin_lower = bin_boundaries[i]
        bin_upper = bin_boundaries[i + 1]
        
        # Include upper boundary for the last bin
        if i == n_bins - 1:
            in_bin = (y_prob >= bin_lower) & (y_prob <= bin_upper)
        else:
            in_bin = (y_prob >= bin_lower) & (y_prob < bin_upper)
            
        prop_in_bin = in_bin.mean()
        if prop_in_bin > 0:
            accuracy_in_bin = y_true[in_bin].mean()
            avg_confidence_in_bin = y_prob[in_bin].mean()
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
    return ece
# ---------------------------------------

SEEDS = [42, 43, 44, 45, 46]
OUTPUT_CSV = "./results.csv"
learn_rate = 1e-4

all_curve_rows = []
all_metrics = []
all_uncertainty_stats = []

# Metrics (Trainer logging only)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

# Bayesian MC Prediction
def bayesian_predict(trainer, dataset, mc_samples=20):
    model = trainer.model
    model.eval()

    probs_all = []

    with torch.no_grad():
        for _ in range(mc_samples):
            outputs = trainer.predict(dataset)
            logits = torch.tensor(outputs.predictions)
            probs = torch.softmax(logits, dim=-1)
            probs_all.append(probs.unsqueeze(0))

    probs_all = torch.cat(probs_all, dim=0)
    mean_probs = probs_all.mean(dim=0)
    uncertainty = probs_all.var(dim=0).mean(dim=1)
    preds = mean_probs.argmax(dim=-1)

    return preds.numpy(), mean_probs.numpy(), uncertainty.numpy()

# Rejection curve computation
def get_curve_data(preds, labels, uncertainty, steps=51):
    sorted_idx = np.argsort(uncertainty)[::-1]
    preds = preds[sorted_idx]
    labels = labels[sorted_idx]

    total = len(labels)
    rates = np.linspace(0, 50, steps)
    accuracies = []

    for r in rates:
        n_rej = int(total * (r / 100.0))
        if n_rej < total:
            acc = np.mean(preds[n_rej:] == labels[n_rej:])
        else:
            acc = 1.0
        accuracies.append(acc)

    # boosted accuracy at 20% rejection
    idx_20 = (np.abs(rates - 20)).argmin()
    boosted_acc_20 = accuracies[idx_20]

    return rates, accuracies, boosted_acc_20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

for seed in SEEDS:
    print("\n" + "=" * 40)
    print(f"STARTING RUN WITH SEED: {seed}")
    print("=" * 40)

    set_seed(seed)

    embedding_dim = train_embeds.shape[1]  # e.g. 768

    model = BayesianEmbeddingClassifier(
        embedding_dim=embedding_dim,
        num_labels=2,
        beta=1e-4
    )

    training_args = TrainingArguments(
        output_dir=f"./results/seed_{seed}",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learn_rate,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=64,
        num_train_epochs=20,
        logging_steps=10,
        load_best_model_at_end=True,
        seed=seed,
        remove_unused_columns=False,
        disable_tqdm=True,
        log_level="error",
        logging_strategy="no",
        save_total_limit=1
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collate_fn,   # IMPORTANT
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    # Bayesian inference on TEST
    preds, probs, uncertainty = bayesian_predict(trainer, test_dataset, mc_samples=20)
    labels = np.array(test_labels)

    # ---- NEW: Calculate ECE and Brier Score ----
    # Extract predicted probability for the positive class (class 1)
    prob_class_1 = probs[:, 1]
    
    brier = brier_score_loss(labels, prob_class_1)
    ece = compute_ece(labels, prob_class_1, n_bins=10)
    # --------------------------------------------

    # ---- NEW: ROC-AUC and AUPRC (Average Precision) ----
    try:
        roc_auc = roc_auc_score(labels, prob_class_1)
    except ValueError:
        # happens if a fold/seed's test labels are single-class
        roc_auc = float("nan")

    try:
        auprc = average_precision_score(labels, prob_class_1)
    except ValueError:
        auprc = float("nan")
    # ------------------------------------------------------

    # ---- Uncertainty diagnostics ----
    correct_mask = (preds == labels)
    incorrect_mask = (preds != labels)

    # Guard against empty slices (rare but safe)
    avg_unc_correct = uncertainty[correct_mask].mean() if correct_mask.any() else 0.0
    avg_unc_incorrect = uncertainty[incorrect_mask].mean() if incorrect_mask.any() else 0.0

    safety_ratio = (
        avg_unc_incorrect / avg_unc_correct
        if avg_unc_correct > 0 else 0.0
    )

    # Per-seed test metrics (MC-mean predictions)
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds)
    rec  = recall_score(labels, preds)
    f1   = f1_score(labels, preds)
    mcc  = matthews_corrcoef(labels, preds)

    rates, accuracies, boosted_acc = get_curve_data(preds, labels, uncertainty)

    all_metrics.append({
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "mcc": mcc,
        "ece": ece,
        "brier": brier,
        "roc_auc": roc_auc,     # <-- Added
        "auprc": auprc,         # <-- Added
        "boosted_acc_20": boosted_acc,
        "gain": boosted_acc - acc,
        "avg_unc_correct": avg_unc_correct,
        "avg_unc_incorrect": avg_unc_incorrect,
        "safety_ratio": safety_ratio
    })

    print(
        f"Seed {seed} | "
        f"Acc: {acc:.4f} | "
        f"F1: {f1:.4f} | "
        f"ECE: {ece:.4f} | "
        f"Brier: {brier:.4f} | "
        f"ROC-AUC: {roc_auc:.4f} | "   # <-- Added
        f"AUPRC: {auprc:.4f}"          # <-- Added
    )

    for r, a in zip(rates, accuracies):
        all_curve_rows.append({
            "seed": seed,
            "rejection_rate": r,
            "accuracy": a
        })

    del model, trainer, preds, probs, uncertainty
    torch.cuda.empty_cache()
    gc.collect()

# Save rejection-curve CSV
df_curves = pd.DataFrame(all_curve_rows)
df_curves.to_csv(OUTPUT_CSV, index=False)
print(f"\nRejection curve data saved to: {OUTPUT_CSV}")

# Final mean ± std over seeds
df_metrics = pd.DataFrame(all_metrics)

print("\n" + "=" * 60)
print(f"FINAL TEST RESULTS (Mean ± Std over {len(SEEDS)} Seeds)")
print("=" * 60)

def print_stat(name, col):
    mean = df_metrics[col].mean()
    std = df_metrics[col].std()
    print(f"{name:<28} | {mean:.4f} ± {std:.4f}")

print_stat("Accuracy", "accuracy")
print_stat("Precision", "precision")
print_stat("Recall", "recall")
print_stat("F1", "f1")
print_stat("MCC", "mcc")
print("=" * 60)
# Uncertainty metrics
print_stat("Avg Uncertainty (Correct)", "avg_unc_correct")
print_stat("Avg Uncertainty (Incorrect)", "avg_unc_incorrect")
print_stat("Safety Ratio", "safety_ratio")
print_stat("Expected Calibration Error", "ece")
print_stat("Brier Score", "brier")
print_stat("ROC-AUC", "roc_auc")
print_stat("AUPRC (Avg Precision)", "auprc")
print("="*60)

In [ ]:
import numpy as np
import pandas as pd

csv_path = "./results.csv"
df = pd.read_csv(csv_path)

# The rejection rates we care about
targets = [0.0, 20.0, 50.0]

print("\n" + "="*56)
print(f"{'Rejection Rate':<20} | {'Mean Acc':<10} | {'Std Dev':<10}")
print("-" * 56)

for r in targets:
    # Use isclose to be safe with floats
    acc = df[np.isclose(df["rejection_rate"], r)]["accuracy"]
    
    mean = acc.mean()
    std  = acc.std(ddof=1)  # sample std across seeds
    
    print(f"{r:>6.0f}%{'':<13} | {mean*100:>7.2f}%   | ±{std*100:>7.2f}%")

print("="*56 + "\n")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.size': 20,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'axes.labelsize': 21,
    'axes.titlesize': 21,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 16,
    'figure.dpi': 300,
    'lines.linewidth': 2.8,
    'lines.markersize': 8.2
})

csv_path = "./results.csv"
df = pd.read_csv(csv_path)

required_cols = {"seed", "rejection_rate", "accuracy"}
assert required_cols.issubset(df.columns), "CSV format incorrect!"

grouped = df.groupby("rejection_rate")["accuracy"]

mean_curve = grouped.mean().values
std_curve  = grouped.std(ddof=1).values   # sample std
curve_rates = grouped.mean().index.values

# Scalars for reporting
base_acc_mean = df[df["rejection_rate"] == 0]["accuracy"].mean()
base_acc_std  = df[df["rejection_rate"] == 0]["accuracy"].std(ddof=1)

acc_20 = df[df["rejection_rate"] == 20]["accuracy"]
boosted_acc_mean = acc_20.mean()
boosted_acc_std  = acc_20.std(ddof=1)

gain_mean = boosted_acc_mean - base_acc_mean
gain_std  = np.sqrt(base_acc_std**2 + boosted_acc_std**2)

print("\n" + "="*56)
print(f"{'Metric':<30} | {'Mean':<10} | {'Std Dev':<10}")
print("-" * 56)

print(f"{'Base Accuracy (0% Rej)':<30} | {base_acc_mean*100:.2f}%     | ±{base_acc_std*100:.2f}%")
print(f"{'Boosted Accuracy (20% Rej)':<30} | {boosted_acc_mean*100:.2f}%     | ±{boosted_acc_std*100:.2f}%")
print(f"{'Net Gain':<30} | {gain_mean*100:.2f}%     | ±{gain_std*100:.2f}%")
print("="*56 + "\n")

plt.figure(figsize=(7, 5))

# A. Shaded Std Region
plt.fill_between(
    curve_rates,
    mean_curve - std_curve,
    mean_curve + std_curve,
    color='#1f77b4',
    alpha=0.2,
    label='Standard Deviation (±1$\sigma$)'
)

# B. Mean Curve
plt.plot(
    curve_rates,
    mean_curve,
    color='#1f77b4',
    linewidth=2.5,
    marker='o',
    markersize=5,
    markevery=5,
    label='Mean Accuracy'
)

# C. 20% Rejection Marker
plt.axvline(
    x=20,
    color='r',
    linestyle='--',
    alpha=0.8,
    linewidth=1.5,
    label='20% Rejection Threshold'
)

# D. Formatting
plt.xlabel("Rejection Rate (%)")
plt.ylabel("Accuracy on Retained Samples")
# plt.title("")
plt.legend(loc="lower right", frameon=True, framealpha=0.95, edgecolor='gray')
plt.grid(True, linestyle=':', alpha=0.6)
plt.xlim(0, 50)

# Save if needed
plt.show()